# LoL ingestion quality

Review the latest source reconciliation, filtering, season coverage, and row-level data completeness. This notebook reads generated artifacts only; ingestion logic remains in `lol_bets`.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").exists())
QUALITY = ROOT / "reports/lol/ingestion/data_quality.json"
RAW = ROOT / "data/lol/raw/raw_data.parquet"
quality = json.loads(QUALITY.read_text())
raw = pd.read_parquet(RAW)
print(f"Rows: {len(raw):,} | games: {raw['gameid'].nunique():,} | columns: {len(raw.columns)}")
quality

In [ ]:
coverage = (raw.assign(year=pd.to_datetime(raw['date']).dt.year)
            .groupby(['year', 'league'], observed=True)['gameid'].nunique()
            .rename('games').reset_index())
display(coverage.sort_values(['year', 'games'], ascending=[False, False]))
coverage.groupby('year')['games'].sum().plot.bar(title='Unique games by season', ylabel='games')
plt.tight_layout()

In [ ]:
reconciliation = pd.DataFrame(quality.get('column_reconciliation', []))
if reconciliation.empty:
    print('No column reconciliation table was recorded.')
else:
    display(pd.crosstab(reconciliation['source_status'], reconciliation['disposition']))
    display(reconciliation.query("source_status != 'present'"))